# Email Phishing Analyzer

This notebook demonstrates how to use the NeMo Agent Toolkit SDK to create an email phishing detection workflow. It covers:

1. **Basic Setup** - Creating the phishing analyzer workflow
2. **LLM Selection** - Switching between different LLM models
3. **Evaluation** - Running evaluations on the workflow
4. **Optimization** - Optimizing workflow parameters

## Prerequisites

- Install the `nat_email_phishing_analyzer` package: `pip install -e examples/evaluation_and_profiling/email_phishing_analyzer`
- Set the `NVIDIA_API_KEY` environment variable


In [ ]:
import os
import sys

# Add src to path for development
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)


## LLM Configuration

Select which LLM to use for the phishing analyzer. Available options:
- `llama-3.1-405b-instruct` - Largest, most capable (default)
- `llama-3.3-70b-instruct` - Good balance of speed and quality
- `llama-3.1-8b-instruct` - Fastest, smallest
- `mixtral-8x22b-instruct` - Mixtral MoE model
- `phi-3-medium-4k-instruct` - Microsoft Phi-3 medium


In [ ]:
# Select the LLM model to use
# Change this to switch between different models
SELECTED_MODEL = "meta/llama-3.3-70b-instruct"

# Available models for easy switching
AVAILABLE_MODELS = {
    "llama-405b": "meta/llama-3.1-405b-instruct",
    "llama-70b": "meta/llama-3.3-70b-instruct",
    "llama-8b": "meta/llama-3.1-8b-instruct",
    "mixtral": "mistralai/mixtral-8x22b-instruct-v0.1",
    "phi-3": "microsoft/phi-3-medium-4k-instruct",
}

print(f"Selected model: {SELECTED_MODEL}")
print(f"\nAvailable models: {list(AVAILABLE_MODELS.keys())}")


## Creating the Workflow

The email phishing analyzer uses a ReAct agent with a custom email analysis tool.


In [ ]:
from pathlib import Path

from nat.agent.react_agent.register import NatReActAgent
from nat.data_models.component_ref import LLMRef
from nat.llm.nim_llm import NimLLM
from nat.utils.sdk.nat_function import NatFunction
from nat.utils.sdk.nat_workflow import NatWorkflow

# Import the custom phishing analyzer (you must install the package first)
# pip install -e examples/evaluation_and_profiling/email_phishing_analyzer
from nat_email_phishing_analyzer.register import EmailPhishingAnalyzerConfig

# Define the phishing analysis prompt
PHISHING_PROMPT = """
Examine the following email content and determine if it exhibits signs of malicious intent. Look for any
suspicious signals that may indicate phishing, such as requests for personal information or suspicious tone.

Email content:
{body}

Return your findings as a JSON object with these fields:

- is_likely_phishing: (boolean) true if phishing is suspected
- explanation: (string) detailed explanation of your reasoning
"""

# Create the main LLM for the workflow
main_llm = NimLLM(
    model_name=SELECTED_MODEL,
    temperature=0.0,
    max_tokens=512,
    name="nim_llm",
)


# Create a wrapper class for the phishing analyzer
class EmailPhishingAnalyzer(EmailPhishingAnalyzerConfig, NatFunction):
    """SDK wrapper for EmailPhishingAnalyzerConfig"""
    pass

# Create the email phishing analyzer tool
email_analyzer = EmailPhishingAnalyzer(
    llm=LLMRef(value=main_llm.computed_name),
    prompt=PHISHING_PROMPT,
    name="email_phishing_analyzer",
)

# Create the ReAct agent that uses the phishing analyzer
agent = NatReActAgent(
    tools=[email_analyzer],
    llm=main_llm,
    verbose=True,
    parse_agent_response_max_retries=3,
)

# Wrap in NatWorkflow
nat_workflow = NatWorkflow(
    entrypoint=agent,
)

print("Workflow created successfully!")


## Testing the Workflow

Let's test the workflow with a sample phishing email and a benign email.


In [ ]:
# Test with a phishing email
phishing_email = """
Dear Customer,

We have detected unusual activity on your account. To prevent suspension,
please verify your identity immediately by clicking the link below and
providing your account credentials, social security number, and credit card details.

URGENT: Your account will be suspended within 24 hours if not verified.

Click here: http://suspicious-link.com/verify

Best regards,
Your Bank Security Team
"""

result = await nat_workflow.prompt(phishing_email)
print("Phishing Email Analysis:")
print(result)


In [ ]:
# Test with a benign email
benign_email = """
Hi John,

Thanks for attending the meeting yesterday. I've attached the notes from our discussion.
Let me know if you have any questions about the project timeline we discussed.

Best,
Sarah
"""

result = await nat_workflow.prompt(benign_email)
print("Benign Email Analysis:")
print(result)


## Saving the Configuration

Save the workflow configuration to a YAML file.


In [ ]:
config_dir = Path(os.getcwd()) / "config"
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "workflow_config.yaml"
nat_workflow.save_to_config_file(config_path)

print(f"Configuration saved to: {config_path}")
print("\n" + "="*50 + "\n")

with open(config_path) as f:
    print(f.read())


## Evaluation

Set up evaluation with the phishing email dataset and run evaluations using Ragas metrics.


In [ ]:
from nat.data_models.dataset_handler import EvalDatasetStructureConfig
from nat.eval.rag_evaluator.register import RagasEvaluator
from nat.utils.sdk.nat_evaluation import EvalDatasetCsvConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation

# Path to the evaluation dataset
dataset_path = Path(os.getcwd()).parent / "data" / "smaller_test.csv"

# Create evaluator LLM (can use a different/smaller model for evaluation)
eval_llm = NimLLM(
    model_name="meta/llama-3.1-70b-instruct",
    temperature=0.0,
    max_tokens=8,
    name="eval_llm",
)

# Create evaluators
accuracy_evaluator = RagasEvaluator(
    llm=eval_llm,
    metric="AnswerAccuracy",
    name="accuracy",
)

groundedness_evaluator = RagasEvaluator(
    llm=eval_llm,
    metric="ResponseGroundedness",
    name="groundedness",
)

relevance_evaluator = RagasEvaluator(
    llm=eval_llm,
    metric="ContextRelevance",
    name="relevance",
)

# Create evaluation configuration with CSV dataset
# The structure config maps CSV columns to expected fields
evaluation = NatEvaluation(
    output_dir=Path(".tmp/nat/examples/email_phishing_analyzer/eval"),
    max_concurrency=4,
    dataset=EvalDatasetCsvConfig(
        file_path=dataset_path,
        id_key="subject",
        structure=EvalDatasetStructureConfig(
            question_key="body",
            answer_key="label",
        ),
    ),
    evaluators=[accuracy_evaluator, groundedness_evaluator, relevance_evaluator],
)

nat_workflow.add_evaluator(evaluation)
print("Evaluation configured!")


In [ ]:
# Save the configuration with evaluation settings
eval_config_path = config_dir / "eval_config.yaml"
nat_workflow.save_to_config_file(eval_config_path)

print(f"Evaluation configuration saved to: {eval_config_path}")
print("\n" + "="*50 + "\n")

with open(eval_config_path) as f:
    print(f.read())


In [ ]:
# Run the evaluation (uncomment to run - this will take some time)
# results = await nat_workflow.evaluate(reps=1)
# print("Evaluation complete!")
# print(results)


## Optimization

Set up and run optimization to find the best parameters for the workflow.


In [ ]:
from nat.utils.sdk.nat_optimizer import NatOptimizer
from nat.utils.sdk.nat_optimizer import NumericOptimizationConfig
from nat.utils.sdk.nat_optimizer import OptimizerMetric

# Create optimizer configuration
optimizer = NatOptimizer(
    output_path=Path(".tmp/nat/examples/email_phishing_analyzer/optimizer"),
    eval_metrics={
        "accuracy": OptimizerMetric(
            evaluator_name="accuracy",
            direction="maximize",
        ),
    },
    reps_per_param_set=1,
    numeric=NumericOptimizationConfig(
        enabled=True,
        n_trials=5,
    ),
)

nat_workflow.add_optimizer(optimizer)
print("Optimizer configured!")


In [ ]:
# Save the configuration with optimizer settings
optimizer_config_path = config_dir / "optimizer_config.yaml"
nat_workflow.save_to_config_file(optimizer_config_path)

print(f"Optimizer configuration saved to: {optimizer_config_path}")
print("\n" + "="*50 + "\n")

with open(optimizer_config_path) as f:
    print(f.read())


In [ ]:
# Run the optimization (uncomment to run - this will take some time)
# optimized_config = await nat_workflow.optimize()
# print("Optimization complete!")
# print(optimized_config)


## Model Comparison

This section shows how to quickly compare different LLMs on the same task.


In [ ]:
async def test_model(model_name: str, test_email: str) -> str:
    """Test a specific model on an email."""
    llm = NimLLM(
        model_name=model_name,
        temperature=0.0,
        max_tokens=512,
        name="test_llm",
    )

    analyzer = EmailPhishingAnalyzer(
        llm=LLMRef(value=llm.computed_name),
        prompt=PHISHING_PROMPT,
        name="test_analyzer",
    )

    test_agent = NatReActAgent(
        tools=[analyzer],
        llm=llm,
        verbose=False,
        parse_agent_response_max_retries=3,
    )

    workflow = NatWorkflow(entrypoint=test_agent)
    return await workflow.prompt(test_email)


# Test email for comparison
comparison_test_email = """
URGENT: Your account has been compromised!
Click here immediately to reset your password: http://fake-bank.com/reset
If you don't act within 1 hour, your funds will be at risk.
"""

# Compare two models (uncomment to run)
# print("Testing llama-70b...")
# result_70b = await test_model("meta/llama-3.3-70b-instruct", comparison_test_email)
# print(f"Llama-70b result: {result_70b}\n")

# print("Testing llama-8b...")
# result_8b = await test_model("meta/llama-3.1-8b-instruct", comparison_test_email)
# print(f"Llama-8b result: {result_8b}")


## Summary

This notebook demonstrated:

1. **Workflow Creation** - Using `NatWorkflow` with `NatReActAgent` and custom tools
2. **LLM Selection** - Easy switching between different NIM models
3. **Evaluation** - Using `NatEvaluation` with Ragas evaluators
4. **Optimization** - Using `NatOptimizer` for parameter tuning
5. **Model Comparison** - Comparing different LLMs on the same task

### Next Steps

- Run the evaluation on the full dataset
- Use optimization to find the best model and parameters
- Try the reasoning agent variant (config-reasoning.yml)
